In [5]:
import re
from pyspark.sql import types as T

In [6]:
%run ./schemas.ipynb

In [7]:
import os

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [8]:
from spark_session import get_spark

In [10]:
spark = get_spark("bad_records")
with_bad = T.StructType(ORDERS.fields + [T.StructField("_corrupt", T.StringType())])
bad_orders = spark.read.csv("data/bad_orders.csv", header=True, schema=with_bad)
bad_orders.show(truncate=False)

bad_orders.cache()
bad_orders.where(bad_orders._corrupt.isNotNull()).select("order_id", "_corrupt").show(truncate=False)
try:
    spark.read.csv("data/bad_orders.csv", header=True, schema=ORDERS, mode="FAILFAST").collect()
except Exception as e:
    print("FAILFAST stopped the job: ", re.search(r"\[(MALFORMED)\w+", str(e)).group(1))

+--------+-------------------+-----------+----------+--------+------+-------------------+--------+
|order_id|order_ts           |customer_id|product_id|quantity|status|updated_at         |_corrupt|
+--------+-------------------+-----------+----------+--------+------+-------------------+--------+
|1       |2026-10-01 10:00:00|5          |P001      |2       |paid  |2026-10-01 10:00:00|NULL    |
|2       |2026-10-01 11:00:00|6          |P002      |NULL    |paid  |2026-10-01 11:00:00|NULL    |
|3       |NULL               |7          |P003      |1       |paid  |2026-10-01 12:00:00|NULL    |
+--------+-------------------+-----------+----------+--------+------+-------------------+--------+

+--------+--------+
|order_id|_corrupt|
+--------+--------+
+--------+--------+

FAILFAST stopped the job:  MALFORMED


In [11]:
spark.stop()